In [ ]:
from pathlib import Path
import os
from pathlib import Path
import sys
sys.path.append(os.path.abspath(".."))
from scripts.Explainer import Explainer
from scripts.Config_Kw_Dict import get_kw_dict
import torch
import rbo
import numpy as np
import pickle


In [2]:
data_name = "ML1M" # Can be ML1M, Yahoo, Pinterest
recommender_name = "MLP" # Can be MLP, VAE
kw_dict=get_kw_dict()


In [ ]:
DP_DIR = Path("scripts", 'checkpoints') 
export_dir = Path(os.getcwd()).parent
files_path = Path(export_dir, DP_DIR)
## path for loading LXR explainer
lxr_path=f'LXR-rep_{data_name}_{recommender_name}_{49}.pt'

## path for loading the records
record_path=f'Records_LF_{data_name}_{recommender_name}.pkl' 

In [ ]:
with open(path(files_path,f'Records_LF_{data_name}_{recommender_name}.pkl'), 'rb') as file:
    record_LF = pickle.load(file)




## Loading LXR explainer

In [ ]:
exp_hid_szie=kw_dict['Predefined_hyperparameters'][recommender_name][data_name]['explainer_hidden_size']
num_items=kw_dict['num_items'][data_name]


def load_explainer():
    explainer = Explainer(num_items, num_items, exp_hid_szie)
    lxr_checkpoint = torch.load(Path(files_path, lxr_path), map_location=torch.device("mps" if torch.backends.mps.is_available() else "cpu"))

    explainer.load_state_dict(lxr_checkpoint)
    explainer.eval()
    for param in explainer.parameters():
        param.requires_grad= False
    return explainer

explainer = load_explainer()

## Rank Bias overlaped

In [ ]:
RBO_list=[]
Mask_LF=record_LF['mask']
Mask_LXR=record_LXR['mask']

for i in range(len(Mask)):

    LXR_Mask=[k for k , v in Mask_LXR[i][0:record_LXR['MPNR'][i]]]
    LF_Mask=[k for k , v in Mask_LF[i][0:record_LF['MPNR'][i]]]

    LF_Mask = [int(t.to("cpu")) for t in LF_Mask]
    RBO_list.append(rbo.RankingSimilarity(LXR_Mask, LF_Mask).rbo())